# Analyze chemical ordering in multi-sublattice materials

This tutorial evaluates supplied occupations; it does not optimize them. It requires
ASE and NumPy, but neither ATAT nor icet. We create reproducible random decorations
of three crystallographic prototypes, validate their site restrictions, and report
sublattice-resolved SRO. Finite random cells need not have zero SRO.

Lattice parameters and the MAX internal coordinate below are illustrative, not
relaxed predictions. These examples make no phase-stability claim.


In [ ]:
from pathlib import Path
import sys
import tempfile
import json
import numpy as np
from ase.spacegroup import crystal

# Works from a checkout or an installed m_scraps package.
for root in (Path.cwd(), Path.cwd().parent):
    if (root / "python/m_scraps").is_dir():
        sys.path.insert(0, str(root / "python"))
        break
from m_scraps import analyze_structure
output = Path(tempfile.mkdtemp(prefix="m_scraps_analysis_"))
print("Results:", output)


## Prototype definitions

* **(Ti,Zr)B₂:** AlB₂ type, P6/mmm (191), M at 1a and B at 2d;
  [AFLOW prototype index](https://aflow.org/p/prototype_index.html), AB2_hP3_191_a_d.
* **(Ti,Zr)₂(Al,Si)C:** MAX-211, P6₃/mmc (194), M at 4f,
  A at 2d, C at 2a. These positions are described in
  [the Ti₂AX structural study](https://www.nature.com/articles/s41598-022-17539-8).
* **(Co,Ni)₂(Mn,Fe)Al:** L2₁, Fm-3m (225), X at 8c, Y at 4a,
  Z at 4b; equivalent to the
  [AFLOW L2₁ prototype](https://aflow.org/p/AB2C_cF16_225_a_c_b-001/)
  under an origin shift exchanging 4a and 4b.

Each conventional cell is repeated 2×2×2. The `spacegroup_kinds` array tracks
which supplied basis representative generated each site; it is independent of
the chemical symbols assigned afterward.


In [ ]:
prototypes = {
    "diboride": dict(symbols=["Ti", "B"], basis=[(0,0,0), (1/3,2/3,.5)],
        spacegroup=191, cellpar=[3.1,3.1,3.3,90,90,120],
        labels=["M", "B"], allowed={"M": ["Ti", "Zr"], "B": ["B"]}),
    "max211": dict(symbols=["Ti", "Al", "C"], basis=[(1/3,2/3,.08), (1/3,2/3,.75), (0,0,0)],
        spacegroup=194, cellpar=[3.05,3.05,13.6,90,90,120],
        labels=["M", "A", "C"], allowed={"M": ["Ti", "Zr"], "A": ["Al", "Si"], "C": ["C"]}),
    "heusler": dict(symbols=["Co", "Mn", "Al"], basis=[(.25,.25,.25), (0,0,0), (.5,.5,.5)],
        spacegroup=225, cellpar=[5.95,5.95,5.95,90,90,90],
        labels=["X", "Y", "Z"], allowed={"X": ["Co", "Ni"], "Y": ["Mn", "Fe"], "Z": ["Al"]}),
}
expected = {
    "diboride": {"Ti":4, "Zr":4, "B":16},
    "max211": {"Ti":16, "Zr":16, "Al":8, "Si":8, "C":16},
    "heusler": {"Co":32, "Ni":32, "Mn":16, "Fe":16, "Al":32},
}


In [ ]:
from collections import Counter
rng = np.random.default_rng(42)
reports = {}
for name, spec in prototypes.items():
    reference = crystal(spec["symbols"], spec["basis"], spacegroup=spec["spacegroup"],
                        cellpar=spec["cellpar"]).repeat((2,2,2))
    labels = np.array([spec["labels"][int(k)] for k in reference.arrays["spacegroup_kinds"]])
    atoms = reference.copy()
    symbols = np.array(atoms.get_chemical_symbols(), dtype=object)
    counts = {}
    for label, alphabet in spec["allowed"].items():
        indices = np.flatnonzero(labels == label)
        assert len(indices) % len(alphabet) == 0
        decoration = np.tile(alphabet, len(indices) // len(alphabet))
        rng.shuffle(decoration)
        symbols[indices] = decoration
        counts[label] = dict(Counter(decoration))
    atoms.set_chemical_symbols(symbols)
    assert dict(Counter(symbols)) == expected[name]
    config = dict(site_labels=labels.tolist(), allowed=spec["allowed"], counts=counts)
    report = analyze_structure(atoms, reference=reference, sublattices=config,
                               cutoff=8, shells=2)
    reports[name] = report
    atoms.write(output / f"{name}.cif")
    reference.write(output / f"{name}_reference.extxyz")
    (output / f"{name}_sublattices.json").write_text(json.dumps(config, indent=2))
    (output / f"{name}_analysis.json").write_text(json.dumps(report, indent=2, allow_nan=False))
    print(name, len(atoms), dict(Counter(symbols)))


## Inspect M–M, A–A, and M–A ordering in the MAX phase

Shell numbers belong to each directed sublattice pair. A shell numbered 1 on
M–M generally has a different distance from shell 1 on M–A. Fixed C sites
remain in the structure and are flagged in the complete report.


In [ ]:
for row in reports["max211"]["sro"]:
    if row["source"] in {"M", "A"} and row["destination"] in {"M", "A"} and row["shell"] == 1:
        print(f"{row['source']}–{row['destination']} "
              f"{row['center_species']}→{row['neighbor_species']}: "
              f"alpha={row['alpha']:.5f}, pairs={row['pair_count']}, "
              f"r={row['distance_min']:.3f} Å")


## Correlations and target errors

For multi-figure analysis, supply a `SCRAPS_FIGURES` file generated for the same
supercell, with the original species-index order:

```python
report = analyze_structure(atoms, cutoff=8, figures="figures.dat",
                           species=["Ti", "Zr", "Al", "Si", "C"])
```

This optional call is not executed here because figure definitions must match
the specific model. It reports every correlation and residual, maximum absolute
error, RMS, mean absolute error and weighted L1. Explicit figure definitions
are essential: pair SRO and decorated cluster correlations are different
observables and their numerical errors cannot be interchanged.

The JSON outputs can be compared in a batch only when reference geometry,
shells/figures, target definitions and weights match.
